# LoRA Fine-Tuning on Ray with Training Hub on OpenShift AI

This notebook runs **LoRA (Low-Rank Adaptation)** fine-tuning using Training Hub on a
Ray cluster. We train [Qwen/Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
to generate SQL queries from natural language questions.

This is the **single GPU** variant — training runs entirely on the head pod.
For multi-GPU distributed LoRA via `TorchTrainer`, see
[`lora_sft-ray-multinode.ipynb`](./lora_sft-ray-multinode.ipynb).

CodeFlare SDK submits the job as a `RayJob` with a `ManagedClusterConfig` — the SDK
creates a short-lived RayCluster, runs the job, and tears everything down on completion.

For more details, see the [README](./README.md).

## Setup

Import the required dependencies.

In [ ]:
from codeflare_sdk import ManagedClusterConfig, RayJob
from kubernetes.client import (
    V1PersistentVolumeClaimVolumeSource,
    V1Volume,
    V1VolumeMount,
)

## Authenticate to your OpenShift Cluster

Log in via `oc login`. Get your token from `oc whoami -t` or the OpenShift console
(Copy login command).

> **Note:** The CodeFlare SDK reads authentication from the local kubeconfig file.
> Using `oc login` ensures the kubeconfig is written correctly for the SDK to pick up.

In [ ]:
!oc login --token=<YOUR_TOKEN> --server=<YOUR_API_SERVER> --insecure-skip-tls-verify

## 1. Configure Training Parameters

Update the following variables to match your environment.

### LoRA Parameters
- **LORA_R**: Rank of the low-rank matrices (higher = more capacity, more memory)
- **LORA_ALPHA**: Scaling factor (typically 2× the rank)
- **MAX_SEQ_LEN**: Maximum sequence length for tokenization
- **LEARNING_RATE**: Learning rate for the optimizer
- **NUM_EPOCHS**: Number of training epochs

In [ ]:
# Cluster settings
NAMESPACE = "<YOUR_NAMESPACE>"
IMAGE = "<RAY_TRAINING_HUB_IMAGE>"

# HuggingFace token (optional — only needed for gated models)
HF_TOKEN = ""

# Storage — PVC with model and training data
PVC_NAME = "<YOUR_PVC_NAME>"
PVC_MOUNT_PATH = "/opt/app-root/src/data"

# Model and data paths (relative to PVC_MOUNT_PATH)
MODEL_PATH = f"{PVC_MOUNT_PATH}/Qwen/Qwen2.5-1.5B-Instruct"
DATA_PATH = f"{PVC_MOUNT_PATH}/datasets/train.jsonl"
CKPT_DIR = f"{PVC_MOUNT_PATH}/checkpoints/lora"

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32
MAX_SEQ_LEN = 512
LEARNING_RATE = 1e-4
NUM_EPOCHS = 1

print(f"Namespace:  {NAMESPACE}")
print(f"Image:      {IMAGE}")
print(f"Model:      {MODEL_PATH}")
print(f"Data:       {DATA_PATH}")

## 2. Configure the Ray Cluster

LoRA runs as a single GPU job — all training happens on the head pod.
LoRA is parameter-efficient and works well even on a single GPU.

The PVC is mounted on the head pod so the model and dataset are accessible.
No worker pods are needed for single GPU LoRA.

In [ ]:
pvc_volume = V1Volume(
    name="training-data",
    persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name=PVC_NAME),
)
pvc_mount = V1VolumeMount(name="training-data", mount_path=PVC_MOUNT_PATH)

env_vars = {}
if HF_TOKEN:
    env_vars["HF_TOKEN"] = HF_TOKEN
    env_vars["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

cluster_config = ManagedClusterConfig(
    image=IMAGE,
    num_workers=0,
    head_cpu_requests=4,
    head_cpu_limits=8,
    head_memory_requests=64,
    head_memory_limits=64,
    head_accelerators={"nvidia.com/gpu": 1},
    volumes=[pvc_volume],
    volume_mounts=[pvc_mount],
    envs=env_vars,
)

print("Cluster config: head pod only (single GPU LoRA)")
print("  GPU: 1× NVIDIA")
print(f"  PVC: {PVC_NAME} mounted at {PVC_MOUNT_PATH}")

## 3. Build the Training Entrypoint

The entrypoint calls `training_hub.lora_sft()` directly.

In [ ]:
entrypoint = f'''python -c "
from training_hub import lora_sft

lora_sft(
    model_path='{MODEL_PATH}',
    data_path='{DATA_PATH}',
    ckpt_output_dir='{CKPT_DIR}',
    num_epochs={NUM_EPOCHS},
    max_seq_len={MAX_SEQ_LEN},
    learning_rate={LEARNING_RATE},
    lora_r={LORA_R},
    lora_alpha={LORA_ALPHA},
)

print('LoRA training completed successfully')
"'''

print("Entrypoint configured")
print(f"  Model: {MODEL_PATH}")
print(f"  Data:  {DATA_PATH}")
print(f"  LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")

## 4. Submit the RayJob

The SDK creates a `RayJob` CR with an embedded `RayCluster` spec. KubeRay:
1. Spins up the Ray cluster (head pod with GPU)
2. Runs the entrypoint on the head node
3. Tears everything down when the job finishes

The `ttl_seconds_after_finished` controls how long the RayJob CR persists after
completion (for log inspection). Set to 0 for immediate cleanup.

In [ ]:
job = RayJob(
    job_name="lora-sft-training-hub",
    entrypoint=entrypoint,
    cluster_config=cluster_config,
    namespace=NAMESPACE,
    ttl_seconds_after_finished=600,
)

job.submit()
print(f"RayJob '{job.name}' submitted to namespace '{NAMESPACE}'")

## 5. Monitor Training Progress

Poll the job status. The status may show as `PENDING` while the RayCluster is
spinning up and pulling the image.

LoRA training is fast — with the default parameters on a single L40S (48GB),
expect approximately 5-15 minutes depending on dataset size.

In [ ]:
job.status()

## 6. View Job Logs

Retrieve the RayJob logs to inspect training output and any errors.

In [ ]:
job.logs()

## 7. Cleanup

The RayCluster is automatically torn down when the job finishes.
The RayJob CR is cleaned up after `ttl_seconds_after_finished` (600 seconds).
Training checkpoints are persisted on the PVC and remain available after cleanup.

Use `job.delete()` for immediate cleanup if needed.

In [ ]:
# Uncomment to delete the RayJob immediately:
# job.delete()

## 8. Test the Trained Model

With the LoRA adapter checkpoints saved to the PVC, we can load the adapter,
merge it with the base model, and test SQL generation directly in this workbench.

In [ ]:
import glob
import os

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint_dirs = sorted(
    glob.glob(os.path.join(CKPT_DIR, "checkpoint-*")), key=os.path.getctime
)
if checkpoint_dirs:
    CHECKPOINTS_PATH = checkpoint_dirs[-1]
else:
    CHECKPOINTS_PATH = CKPT_DIR
print(f"Loading checkpoint: {CHECKPOINTS_PATH}")

print(f"Loading base model from: {MODEL_PATH}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.float32,
)
model = PeftModel.from_pretrained(base_model, CHECKPOINTS_PATH)
model = model.merge_and_unload()
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print("LoRA adapter merged with base model")

In [ ]:
def generate_sql(question: str, schema: str, max_tokens: int = 256) -> str:
    """Generate a SQL query from a natural language question."""
    messages = [
        {
            "role": "user",
            "content": f"""Given the following database schema:

{schema}

Write a SQL query to answer this question: {question}""",
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return response.strip()

In [ ]:
test_examples = [
    {
        "schema": "CREATE TABLE employees (id INT, name VARCHAR, department VARCHAR, salary DECIMAL, hire_date DATE)",
        "question": "What is the average salary of employees in the engineering department?",
    },
    {
        "schema": "CREATE TABLE orders (order_id INT, customer_id INT, product_name VARCHAR, quantity INT, order_date DATE)",
        "question": "How many orders were placed in the last 30 days?",
    },
    {
        "schema": "CREATE TABLE students (student_id INT, name VARCHAR, grade INT, subject VARCHAR, score DECIMAL)",
        "question": "Find the top 5 students with the highest average score across all subjects.",
    },
]

print("Testing the trained model:")
print("=" * 60)

for i, example in enumerate(test_examples, 1):
    print(f"\nExample {i}:")
    print(f"Schema: {example['schema']}")
    print(f"Question: {example['question']}")

    sql = generate_sql(example["question"], example["schema"])
    print(f"Generated SQL: {sql}")
    print("-" * 60)

## Summary

This notebook demonstrated end-to-end LoRA fine-tuning on Ray with Training Hub:

1. **Configured** a single GPU RayJob with PVC storage
2. **Trained** a LoRA adapter on Qwen 2.5 1.5B Instruct for natural language to SQL generation
3. **Evaluated** the merged model on SQL generation examples

The LoRA adapter checkpoints are persisted on the PVC and can be merged with the base
model for serving, or used as a starting point for further fine-tuning.